# 01 — Run Experiment

Runs LLM API calls and saves results to Google Drive.

**First time?** See README.md for setup instructions.

## 1. Clone Repo & Mount Drive

In [1]:
import os

# Reset to a safe directory first
os.chdir('/content')

# Clone repo (fresh each time)
!rm -rf /content/authority-bias-llm
!git clone https://github.com/auertobias/authority-bias-llm.git /content/authority-bias-llm

# Change into the repo directory
os.chdir('/content/authority-bias-llm')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q google-genai openai anthropic

Cloning into '/content/authority-bias-llm'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 78 (delta 40), reused 25 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 352.67 KiB | 3.49 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.9 MB/s eta 0:00:00


## 2. Configure

In [2]:
import os, sys
sys.path.insert(0, '.')

from src.config import DATA_PATH, RESULTS_PATH, N_REPS, PAUSE_SECONDS, CHECKPOINT_EVERY

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"Data saves to: {DATA_PATH}")

Data saves to: /content/drive/MyDrive/PhD/2PhD 1Paper/data/


In [4]:
# ── API Keys ───────────────────────────────────────────────
# Use Colab Secrets (🔑 icon in sidebar) — add your keys there.
from google.colab import userdata

# Uncomment the keys you have:
GEMINI_API_KEY    = userdata.get('GEMINI_API_KEY')
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
OPENAI_API_KEY    = userdata.get('OPEN_API_KEY')
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

## 3. Load Data & Build Trials

In [5]:
import pandas as pd

arguments   = pd.read_csv(DATA_PATH + "arguments.csv")
authorities = pd.read_csv(DATA_PATH + "authority.csv")

print(f"Arguments:   {len(arguments)} rows, {arguments['claim_id'].nunique()} claims")
print(f"Authorities: {len(authorities)} rows")

Arguments:   148 rows, 74 claims
Authorities: 12 rows


In [6]:
trials = []
trial_id = 0

for _, arg in arguments.iterrows():
    for _, auth in authorities.iterrows():
        if auth['branch'] == 'none':
            expertise = 'irrelevant'
        elif auth['branch'] == arg['branch']:
            expertise = 'relevant'
        else:
            expertise = 'irrelevant'

        trial_id += 1
        trials.append({
            'trial_id': trial_id,
            'argument_id': arg['id'],
            'claim': arg['claim'],
            'argument': arg['argument'],
            'stance': arg['stance'],
            'arg_type': arg['type'],
            'arg_branch': arg['branch'],
            'authority_id': auth['authority_id'],
            'authority_label': auth['label'].strip(),
            'status': auth['status'],
            'auth_branch': auth['branch'],
            'expertise': expertise,
        })

trials_df = pd.DataFrame(trials)
trials_df = trials_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total trials: {len(trials_df)}")
print(f"\nTrials per cell:")
print(trials_df.groupby(['status', 'expertise', 'arg_type']).size().unstack(fill_value=0))

Total trials: 1776

Trials per cell:
arg_type              descriptive  normative
status    expertise                         
high      irrelevant          222        222
          relevant             74         74
low       irrelevant          222        222
          relevant             74         74
medium    irrelevant           74         74
          relevant             74         74
very high irrelevant           74         74
          relevant             74         74


## 4. Choose Models & Run

In [7]:
import importlib
import src.models
importlib.reload(src.models)

<module 'src.models' from '/content/authority-bias-llm/src/models.py'>

In [10]:
from src.models import make_gemini_fn, make_gpt_fn, make_claude_fn, make_deepseek_fn
from src.prompts import build_prompt, build_prompt_hidden
from src.parsing import extract_rating

# Uncomment the models you want to run:
MODELS = {
    # 'gemini': make_gemini_fn(GEMINI_API_KEY),
    'gpt':    make_gpt_fn(OPENAI_API_KEY),
    # 'claude': make_claude_fn(ANTHROPIC_API_KEY),
    # 'deepseek': make_deepseek_fn(DEEPSEEK_API_KEY),
}

In [11]:
# Hidden condition: reduced design — high, very high, and low status only
HIDDEN_STATUSES = ['low', 'high', 'very high']

trials_hidden = trials_df[
    trials_df['status'].str.strip().isin(HIDDEN_STATUSES)
].copy().reset_index(drop=True)

print(f"Open trials:   {len(trials_df)}")
print(f"Hidden trials: {len(trials_hidden)}  (low + high + very high only)")

Open trials:   1776
Hidden trials: 1480  (low + high + very high only)


In [12]:
import time
from datetime import datetime

for model_name, model_fn in MODELS.items():

    # ── CONDITION 1: OPEN (full argument visible) ─────────────
    print(f"\n{'='*60}")
    print(f"  RUNNING: {model_name.upper()} — OPEN condition")
    print(f"{'='*60}")

    results_open = []

    for rep in range(N_REPS):
        run_order = trials_df.sample(frac=1, random_state=rep*1000).reset_index(drop=True)
        print(f"\n--- Rep {rep+1}/{N_REPS} ({len(run_order)} trials) ---")

        for idx, row in run_order.iterrows():
            prompt = build_prompt(row)
            raw = model_fn(prompt)
            rating = extract_rating(raw)

            results_open.append({
                'trial_id':        row['trial_id'],
                'rep':             rep + 1,
                'model':           model_name,
                'condition':       'open',
                'status':          row['status'],
                'expertise':       row['expertise'],
                'arg_type':        row['arg_type'],
                'arg_branch':      row['arg_branch'],
                'authority_label': row['authority_label'],
                'stance':          row['stance'],
                'claim':           row['claim'],
                'argument':        row['argument'],
                'rating':          rating,
                'raw_response':    raw,
            })

            n_done = idx + 1
            if n_done % 50 == 0:
                print(f"  {n_done}/{len(run_order)} done | last rating: {rating}")

            if n_done % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results_open).to_csv(
                    DATA_PATH + f"checkpoint_{model_name}_open.csv", index=False
                )

            time.sleep(PAUSE_SECONDS)

    # Save open condition
    date_str = datetime.now().strftime("%Y%m%d")
    filepath_open = DATA_PATH + f"data_{model_name}_open_{date_str}.csv"
    pd.DataFrame(results_open).to_csv(filepath_open, index=False)
    valid = sum(1 for r in results_open if r['rating'] is not None)
    print(f"\n✓ Saved: {filepath_open}")
    print(f"  {len(results_open)} rows, {valid} valid ratings ({100*valid/len(results_open):.0f}%)")

    # ── CONDITION 2: HIDDEN (argument not shown) ──────────────
    print(f"\n{'='*60}")
    print(f"  RUNNING: {model_name.upper()} — HIDDEN condition")
    print(f"{'='*60}")

    results_hidden = []

    for rep in range(N_REPS):
        run_order = trials_hidden.sample(frac=1, random_state=rep*2000).reset_index(drop=True)
        print(f"\n--- Rep {rep+1}/{N_REPS} ({len(run_order)} trials) ---")

        for idx, row in run_order.iterrows():
            prompt = build_prompt_hidden(row)
            raw = model_fn(prompt)
            rating = extract_rating(raw)

            results_hidden.append({
                'trial_id':        row['trial_id'],
                'rep':             rep + 1,
                'model':           model_name,
                'condition':       'hidden',
                'status':          row['status'],
                'expertise':       row['expertise'],
                'arg_type':        row['arg_type'],
                'arg_branch':      row['arg_branch'],
                'authority_label': row['authority_label'],
                'stance':          row['stance'],
                'claim':           row['claim'],
                'argument':        '',
                'rating':          rating,
                'raw_response':    raw,
            })

            n_done = idx + 1
            if n_done % 50 == 0:
                print(f"  {n_done}/{len(run_order)} done | last rating: {rating}")

            if n_done % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results_hidden).to_csv(
                    DATA_PATH + f"checkpoint_{model_name}_hidden.csv", index=False
                )

            time.sleep(PAUSE_SECONDS)

    # Save hidden condition
    filepath_hidden = DATA_PATH + f"data_{model_name}_hidden_{date_str}.csv"
    pd.DataFrame(results_hidden).to_csv(filepath_hidden, index=False)
    valid = sum(1 for r in results_hidden if r['rating'] is not None)
    print(f"\n✓ Saved: {filepath_hidden}")
    print(f"  {len(results_hidden)} rows, {valid} valid ratings ({100*valid/len(results_hidden):.0f}%)")

    print(f"\n{'='*60}")
    print(f"  DONE: {model_name.upper()} — both conditions complete")
    print(f"{'='*60}")


  RUNNING: GPT — OPEN condition

--- Rep 1/1 (1776 trials) ---
  50/1776 done | last rating: 60
  100/1776 done | last rating: 60
  150/1776 done | last rating: 70
  200/1776 done | last rating: 70
  250/1776 done | last rating: 60
  300/1776 done | last rating: 70
  350/1776 done | last rating: 60
  400/1776 done | last rating: 60
  450/1776 done | last rating: 60
  500/1776 done | last rating: 60
  550/1776 done | last rating: 70
  600/1776 done | last rating: 60
  650/1776 done | last rating: 60
  700/1776 done | last rating: 60
  750/1776 done | last rating: 40
  800/1776 done | last rating: 85
  850/1776 done | last rating: 60
  900/1776 done | last rating: 40
  950/1776 done | last rating: 85
  1000/1776 done | last rating: 65
  1050/1776 done | last rating: 60
  1100/1776 done | last rating: 60
  1150/1776 done | last rating: 60
  1200/1776 done | last rating: 85
  1250/1776 done | last rating: 60
  1300/1776 done | last rating: 40
  1350/1776 done | last rating: 85
  1400/1776

## 5. Verify

In [ ]:
import glob

data_files = sorted(glob.glob(DATA_PATH + "data_*.csv"))
print("Data files on Drive:")
for f in data_files:
    df_check = pd.read_csv(f)
    valid = df_check['rating'].notna().sum()
    print(f"  {os.path.basename(f):40s} → {len(df_check)} rows, {valid} valid")

Data files on Drive:
  data_deepseek_20260312.csv               → 1776 rows, 1776 valid
  data_gemini_20260313.csv                 → 1776 rows, 1483 valid
  data_gpt_20260311.csv                    → 1776 rows, 1776 valid
